# EYES-DEFY-ANEMIA -- Step 1: Measurement Harness

Pooled out-of-fold repeated stratified cross-validation for all 12 clean-data CNN combos.
**No model is changed and no intervention is applied** -- this run exists solely to replace a
statistically unusable estimator with a usable one, and to produce the frozen baseline that
Steps 2-5 are measured against.

**The problem being fixed.** The single 70/15/15 split leaves 14 India validation patients
(10 anemic / 4 healthy), so India AUC is computed over 10x4 = 40 discordant pairs -- a 95% CI
half-width of roughly +/-0.27. Under that noise floor the observed India AUC spread across the
12 combos (0.550-1.000) is not a ranking, and no later intervention could be shown to work.

**What this run produces.** 5-fold x 5-repeat CV over the 184-patient train+val pool, stratified
on the compound country x label key. India pairs go from **40 to 1,311**; Italy from **60 to 1,680**
(palpebral; forniceal_palpebral is 1,311 / 1,501 -- all 6 patients missing a forniceal crop are Italy).

**The 33-patient test split is sealed** and is asserted absent from every fold. It is spent exactly
once, in Step 6.

Runtime note: 12 combos x 25 fits. `sync_outputs()` runs after every combo, so an interrupted
session still yields a downloadable zip of everything completed so far.

**Companion notebook:** `step1-cv-harness-vit.ipynb` covers the remaining 6 transformer
combos (`swin_t`, `vit_b_16`, `vit_l_16` x both tissue types) separately -- run both, then merge
their downloaded `outputs/` locally before running `aggregate_baseline.py` to get the full
18-combo Step 1 baseline. This notebook alone produces a 12-combo (CNN-only) baseline.

## Setup

In [1]:
import torch

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA available: True
GPU: Tesla T4


In [2]:
# rm -rf first so a re-run within the same kernel session stays idempotent
# instead of nesting a second clone inside the first (this bit an earlier
# version of this notebook during interactive debugging).
!rm -rf eyes-defy-anemia
!git clone https://github.com/manivafapour/eyes-defy-anemia.git
%cd eyes-defy-anemia

Cloning into 'eyes-defy-anemia'...
remote: Enumerating objects: 940, done.
remote: Counting objects: 100% (621/621), done.
remote: Compressing objects: 100% (415/415), done.
remote: Total 940 (delta 274), reused 531 (delta 203), pack-reused 319 (from 1)
Receiving objects: 100% (940/940), 76.25 MiB | 39.42 MiB/s, done.
Resolving deltas: 100% (437/437), done.
/kaggle/working/eyes-defy-anemia


In [3]:
# Diagnostic only -- kept for visibility in the saved run log. The path used
# below is already confirmed correct, so this cell doesn't gate anything, but
# it's cheap and makes a future path change easy to spot in the output.
import os

for name in os.listdir("/kaggle/input"):
    path = f"/kaggle/input/{name}"
    print(name, "->", os.listdir(path))

datasets -> ['manivafapour33']


In [4]:
# Only packages actually missing from Kaggle's base image. Deliberately NOT
# `pip install -r requirements.txt` -- that file is pinned to the local
# Windows/CUDA 13.0 build and would try to reinstall Kaggle's own correctly
# configured GPU PyTorch with an incompatible build.
!pip install -q optuna albumentations

## Data

In [5]:
import shutil
from pathlib import Path

# TODO: verify against cell 4's /kaggle/input listing before running -- this is a best-guess
# default following the established manivafapour33/<slug> pattern, NOT yet confirmed for the
# new clean-data dataset (see classification/.project_memory/kaggle/01_kaggle_notes.md).
SRC_DIR = Path("/kaggle/input/datasets/manivafapour33/processed-dataset-clean")
DST_DIR = Path("classification/data/processed")

# Clear the destination first so this cell is fully idempotent -- a re-run (or
# the two messy copy attempts from the earlier notebook version) never leaves
# stale or duplicated content behind. DST_DIR always ends up as an exact,
# deterministic copy of SRC_DIR, nothing more.
shutil.rmtree(DST_DIR, ignore_errors=True)
DST_DIR.mkdir(parents=True, exist_ok=True)

for item in SRC_DIR.iterdir():
    dest = DST_DIR / item.name
    if item.is_dir():
        shutil.copytree(item, dest)
    else:
        shutil.copy2(item, dest)

print("classification/data/processed now contains:")
for sub in sorted(DST_DIR.iterdir()):
    if sub.is_dir():
        n_files = sum(1 for f in sub.rglob("*") if f.is_file())
        print(f"  {sub.name}/  ({n_files} files)")
    else:
        print(f"  {sub.name}")

classification/data/processed now contains:
  extraction_log.csv
  images/  (428 files)
  metadata.csv
  splits.csv


## Structural verification -- checkpoints 1-4 and 6

Trains nothing; runs in seconds. Fold geometry is where a silent error would be most damaging
and least visible: a leaked test patient or a fold holding only 2 India-healthy patients would
not raise, it would quietly yield a plausible-looking wrong number.

**This cell gates the run.** A non-zero exit means do not proceed to training.

In [6]:
!python classification/step1_cv_harness/validate_harness.py


=== Structural verification: tissue_type=palpebral (k=5, repeats=5, seed=42) ===
  [PASS] 1. test-set isolation: 0 of 33 held-out test patients appear in any fold
  [PASS] 2. partition integrity: each of 5 repeats partitions all 184 pool patients exactly once
  [PASS] 3. rare-cell coverage: min India_0 per outer val fold = 4 (at r0f0), min Italy_1 per outer val fold = 4 (at r0f0)
  [PASS] 4. pair counts (palpebral): pool n=184 cells={'India_0': 23, 'India_1': 57, 'Italy_0': 84, 'Italy_1': 20} | India 57x23=1311 pairs (32.8x the single split's 40) | Italy 20x84=1680 pairs (28.0x the single split's 60)
  [PASS] 6. fold reproducibility: two independent build_folds(seed=42) calls produced identical assignments
  --> ALL STRUCTURAL CHECKS PASSED

=== Structural verification: tissue_type=forniceal_palpebral (k=5, repeats=5, seed=42) ===
  [PASS] 1. test-set isolation: 0 of 33 held-out test patients appear in any fold
  [PASS] 2. partition integrity: each of 5 repeats partitions all 178 pool

In [7]:
# Locked hyperparameters, read straight out of each combo's own v2_clean study summary
# (never hand-transcribed). No re-tuning happens in Step 1 -- re-tuning inside the new
# protocol would confound 'better measurement' with 'better hyperparameters'.
!python classification/step1_cv_harness/run_cv_harness.py --list

18 combos discovered (source: each combo's own v2_clean study summary):

  convnext_tiny_forniceal_palpebral_v2_clean
      arch=convnext_tiny        tissue=forniceal_palpebral  lr=0.0773445 wd=0.000479821 dropout=0.5
  convnext_tiny_palpebral_v2_clean
      arch=convnext_tiny        tissue=palpebral            lr=0.000585494 wd=2.00941e-05 dropout=0.2
  densenet121_forniceal_palpebral_v2_clean
      arch=densenet121          tissue=forniceal_palpebral  lr=0.0183628 wd=0.000153747 dropout=0.5
  densenet121_palpebral_v2_clean
      arch=densenet121          tissue=palpebral            lr=0.000293803 wd=2.93754e-06 dropout=0.5
  efficientnet_b0_forniceal_palpebral_v2_clean
      arch=efficientnet_b0      tissue=forniceal_palpebral  lr=0.00132929 wd=0.000711448 dropout=0.2
  efficientnet_b0_palpebral_v2_clean
      arch=efficientnet_b0      tissue=palpebral            lr=0.0314288 wd=4.33528e-06 dropout=0.5
  mobilenet_v3_small_forniceal_palpebral_v2_clean
      arch=mobilenet_v3_small   

## Output syncing

In [8]:
import shutil
from pathlib import Path


def sync_outputs():
    """Consolidate classification/step1_cv_harness/outputs/ into
    /kaggle/working/outputs/ and re-zip to step1_cv_results_cnn.zip. Called after
    EVERY combo, not just at the end -- this is the longest job this project has
    attempted, so whatever has completed must always be downloadable."""
    results_dir = Path("/kaggle/working/outputs")
    results_dir.mkdir(parents=True, exist_ok=True)
    src = Path("classification/step1_cv_harness/outputs")
    if src.exists():
        shutil.copytree(src, results_dir, dirs_exist_ok=True)
    archive = shutil.make_archive("/kaggle/working/step1_cv_results_cnn", "zip", root_dir=str(results_dir))
    n = sum(1 for f in results_dir.rglob("*") if f.is_file())
    print(f"[sync_outputs] {n} files under {results_dir}, zipped to {archive}")


sync_outputs()  # picks up structural_verification.json; confirms the function works before training

[sync_outputs] 1 files under /kaggle/working/outputs, zipped to /kaggle/working/step1_cv_results_cnn.zip


## Training -- 12 combos, cheapest architecture first

25 fits per combo (5 folds x 5 repeats). Each fold trains on its own inner-split early stopping
and predicts once on its held-out outer fold; the outer fold is never consulted during training
or epoch selection. No checkpoints are written -- Step 1 needs predictions, not weights.

Budget fallback if the session is tight on time: add `--repeats 3` to any cell.

In [9]:
# Step 1 -- 1/12: regnet_y_400mf_palpebral
!python classification/step1_cv_harness/run_cv_harness.py --combo regnet_y_400mf_palpebral_v2_clean
sync_outputs()


Step 1 harness: regnet_y_400mf_palpebral_v2_clean
  device=cuda  arch=regnet_y_400mf  tissue=palpebral
  locked hyperparameters (from v2_clean_scripts/outputs/regnet_y_400mf_palpebral_v2_clean/regnet_y_400mf_palpebral_v2_clean_study_summary.json): lr=0.00112494 wd=0.000118567 dropout=0.2

  Structural verification (checkpoints 1-4):
  [PASS] 1. test-set isolation: 0 of 33 held-out test patients appear in any fold
  [PASS] 2. partition integrity: each of 5 repeats partitions all 184 pool patients exactly once
  [PASS] 3. rare-cell coverage: min India_0 per outer val fold = 4 (at r0f0), min Italy_1 per outer val fold = 4 (at r0f0)
  [PASS] 4. pair counts (palpebral): pool n=184 cells={'India_0': 23, 'India_1': 57, 'Italy_0': 84, 'Italy_1': 20} | India 57x23=1311 pairs (32.8x the single split's 40) | Italy 20x84=1680 pairs (28.0x the single split's 60)

  Training 25 fits (5 repeats x 5 folds):
Downloading: "https://download.pytorch.org/models/regnet_y_400mf-c65dace8.pth" to /root/.cache

In [10]:
# Step 1 -- 2/12: regnet_y_400mf_forniceal_palpebral
!python classification/step1_cv_harness/run_cv_harness.py --combo regnet_y_400mf_forniceal_palpebral_v2_clean
sync_outputs()


Step 1 harness: regnet_y_400mf_forniceal_palpebral_v2_clean
  device=cuda  arch=regnet_y_400mf  tissue=forniceal_palpebral
  locked hyperparameters (from v2_clean_scripts/outputs/regnet_y_400mf_forniceal_palpebral_v2_clean/regnet_y_400mf_forniceal_palpebral_v2_clean_study_summary.json): lr=0.00132929 wd=0.000711448 dropout=0.2

  Structural verification (checkpoints 1-4):
  [PASS] 1. test-set isolation: 0 of 33 held-out test patients appear in any fold
  [PASS] 2. partition integrity: each of 5 repeats partitions all 178 pool patients exactly once
  [PASS] 3. rare-cell coverage: min India_0 per outer val fold = 4 (at r0f0), min Italy_1 per outer val fold = 3 (at r0f4)
  [PASS] 4. pair counts (forniceal_palpebral): pool n=178 cells={'India_0': 23, 'India_1': 57, 'Italy_0': 79, 'Italy_1': 19} | India 57x23=1311 pairs (32.8x the single split's 40) | Italy 19x79=1501 pairs (25.0x the single split's 60)

  Training 25 fits (5 repeats x 5 folds):
    r0f0: stopped epoch 45 (best inner epoch

In [11]:
# Step 1 -- 3/12: mobilenet_v3_small_palpebral
!python classification/step1_cv_harness/run_cv_harness.py --combo mobilenet_v3_small_palpebral_v2_clean
sync_outputs()


Step 1 harness: mobilenet_v3_small_palpebral_v2_clean
  device=cuda  arch=mobilenet_v3_small  tissue=palpebral
  locked hyperparameters (from v2_clean_scripts/outputs/mobilenet_v3_small_palpebral_v2_clean/mobilenet_v3_small_palpebral_v2_clean_study_summary.json): lr=0.0808856 wd=1.0423e-06 dropout=0.5

  Structural verification (checkpoints 1-4):
  [PASS] 1. test-set isolation: 0 of 33 held-out test patients appear in any fold
  [PASS] 2. partition integrity: each of 5 repeats partitions all 184 pool patients exactly once
  [PASS] 3. rare-cell coverage: min India_0 per outer val fold = 4 (at r0f0), min Italy_1 per outer val fold = 4 (at r0f0)
  [PASS] 4. pair counts (palpebral): pool n=184 cells={'India_0': 23, 'India_1': 57, 'Italy_0': 84, 'Italy_1': 20} | India 57x23=1311 pairs (32.8x the single split's 40) | Italy 20x84=1680 pairs (28.0x the single split's 60)

  Training 25 fits (5 repeats x 5 folds):
Downloading: "https://download.pytorch.org/models/mobilenet_v3_small-047dcff4.pt

In [12]:
# Step 1 -- 4/12: mobilenet_v3_small_forniceal_palpebral
!python classification/step1_cv_harness/run_cv_harness.py --combo mobilenet_v3_small_forniceal_palpebral_v2_clean
sync_outputs()


Step 1 harness: mobilenet_v3_small_forniceal_palpebral_v2_clean
  device=cuda  arch=mobilenet_v3_small  tissue=forniceal_palpebral
  locked hyperparameters (from v2_clean_scripts/outputs/mobilenet_v3_small_forniceal_palpebral_v2_clean/mobilenet_v3_small_forniceal_palpebral_v2_clean_study_summary.json): lr=0.00804478 wd=0.000148839 dropout=0.5

  Structural verification (checkpoints 1-4):
  [PASS] 1. test-set isolation: 0 of 33 held-out test patients appear in any fold
  [PASS] 2. partition integrity: each of 5 repeats partitions all 178 pool patients exactly once
  [PASS] 3. rare-cell coverage: min India_0 per outer val fold = 4 (at r0f0), min Italy_1 per outer val fold = 3 (at r0f4)
  [PASS] 4. pair counts (forniceal_palpebral): pool n=178 cells={'India_0': 23, 'India_1': 57, 'Italy_0': 79, 'Italy_1': 19} | India 57x23=1311 pairs (32.8x the single split's 40) | Italy 19x79=1501 pairs (25.0x the single split's 60)

  Training 25 fits (5 repeats x 5 folds):
    r0f0: stopped epoch 9 (b

In [13]:
# Step 1 -- 5/12: efficientnet_b0_palpebral
!python classification/step1_cv_harness/run_cv_harness.py --combo efficientnet_b0_palpebral_v2_clean
sync_outputs()


Step 1 harness: efficientnet_b0_palpebral_v2_clean
  device=cuda  arch=efficientnet_b0  tissue=palpebral
  locked hyperparameters (from v2_clean_scripts/outputs/efficientnet_b0_palpebral_v2_clean/efficientnet_b0_palpebral_v2_clean_study_summary.json): lr=0.0314288 wd=4.33528e-06 dropout=0.5

  Structural verification (checkpoints 1-4):
  [PASS] 1. test-set isolation: 0 of 33 held-out test patients appear in any fold
  [PASS] 2. partition integrity: each of 5 repeats partitions all 184 pool patients exactly once
  [PASS] 3. rare-cell coverage: min India_0 per outer val fold = 4 (at r0f0), min Italy_1 per outer val fold = 4 (at r0f0)
  [PASS] 4. pair counts (palpebral): pool n=184 cells={'India_0': 23, 'India_1': 57, 'Italy_0': 84, 'Italy_1': 20} | India 57x23=1311 pairs (32.8x the single split's 40) | Italy 20x84=1680 pairs (28.0x the single split's 60)

  Training 25 fits (5 repeats x 5 folds):
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" t

In [14]:
# Step 1 -- 6/12: efficientnet_b0_forniceal_palpebral
!python classification/step1_cv_harness/run_cv_harness.py --combo efficientnet_b0_forniceal_palpebral_v2_clean
sync_outputs()


Step 1 harness: efficientnet_b0_forniceal_palpebral_v2_clean
  device=cuda  arch=efficientnet_b0  tissue=forniceal_palpebral
  locked hyperparameters (from v2_clean_scripts/outputs/efficientnet_b0_forniceal_palpebral_v2_clean/efficientnet_b0_forniceal_palpebral_v2_clean_study_summary.json): lr=0.00132929 wd=0.000711448 dropout=0.2

  Structural verification (checkpoints 1-4):
  [PASS] 1. test-set isolation: 0 of 33 held-out test patients appear in any fold
  [PASS] 2. partition integrity: each of 5 repeats partitions all 178 pool patients exactly once
  [PASS] 3. rare-cell coverage: min India_0 per outer val fold = 4 (at r0f0), min Italy_1 per outer val fold = 3 (at r0f4)
  [PASS] 4. pair counts (forniceal_palpebral): pool n=178 cells={'India_0': 23, 'India_1': 57, 'Italy_0': 79, 'Italy_1': 19} | India 57x23=1311 pairs (32.8x the single split's 40) | Italy 19x79=1501 pairs (25.0x the single split's 60)

  Training 25 fits (5 repeats x 5 folds):
    r0f0: stopped epoch 39 (best inner e

In [15]:
# Step 1 -- 7/12: resnet18_palpebral
!python classification/step1_cv_harness/run_cv_harness.py --combo resnet18_palpebral_v2_clean
sync_outputs()


Step 1 harness: resnet18_palpebral_v2_clean
  device=cuda  arch=resnet18  tissue=palpebral
  locked hyperparameters (from v2_clean_scripts/outputs/resnet18_palpebral_v2_clean/resnet18_palpebral_v2_clean_study_summary.json): lr=0.00214548 wd=1.12112e-06 dropout=0.2

  Structural verification (checkpoints 1-4):
  [PASS] 1. test-set isolation: 0 of 33 held-out test patients appear in any fold
  [PASS] 2. partition integrity: each of 5 repeats partitions all 184 pool patients exactly once
  [PASS] 3. rare-cell coverage: min India_0 per outer val fold = 4 (at r0f0), min Italy_1 per outer val fold = 4 (at r0f0)
  [PASS] 4. pair counts (palpebral): pool n=184 cells={'India_0': 23, 'India_1': 57, 'Italy_0': 84, 'Italy_1': 20} | India 57x23=1311 pairs (32.8x the single split's 40) | Italy 20x84=1680 pairs (28.0x the single split's 60)

  Training 25 fits (5 repeats x 5 folds):
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet1

In [16]:
# Step 1 -- 8/12: resnet18_forniceal_palpebral
!python classification/step1_cv_harness/run_cv_harness.py --combo resnet18_forniceal_palpebral_v2_clean
sync_outputs()


Step 1 harness: resnet18_forniceal_palpebral_v2_clean
  device=cuda  arch=resnet18  tissue=forniceal_palpebral
  locked hyperparameters (from v2_clean_scripts/outputs/resnet18_forniceal_palpebral_v2_clean/resnet18_forniceal_palpebral_v2_clean_study_summary.json): lr=0.000293803 wd=2.93754e-06 dropout=0.5

  Structural verification (checkpoints 1-4):
  [PASS] 1. test-set isolation: 0 of 33 held-out test patients appear in any fold
  [PASS] 2. partition integrity: each of 5 repeats partitions all 178 pool patients exactly once
  [PASS] 3. rare-cell coverage: min India_0 per outer val fold = 4 (at r0f0), min Italy_1 per outer val fold = 3 (at r0f4)
  [PASS] 4. pair counts (forniceal_palpebral): pool n=178 cells={'India_0': 23, 'India_1': 57, 'Italy_0': 79, 'Italy_1': 19} | India 57x23=1311 pairs (32.8x the single split's 40) | Italy 19x79=1501 pairs (25.0x the single split's 60)

  Training 25 fits (5 repeats x 5 folds):
    r0f0: stopped epoch 174 (best inner epoch 167, inner_loss=0.480

In [17]:
# Step 1 -- 9/12: densenet121_palpebral
!python classification/step1_cv_harness/run_cv_harness.py --combo densenet121_palpebral_v2_clean
sync_outputs()


Step 1 harness: densenet121_palpebral_v2_clean
  device=cuda  arch=densenet121  tissue=palpebral
  locked hyperparameters (from v2_clean_scripts/outputs/densenet121_palpebral_v2_clean/densenet121_palpebral_v2_clean_study_summary.json): lr=0.000293803 wd=2.93754e-06 dropout=0.5

  Structural verification (checkpoints 1-4):
  [PASS] 1. test-set isolation: 0 of 33 held-out test patients appear in any fold
  [PASS] 2. partition integrity: each of 5 repeats partitions all 184 pool patients exactly once
  [PASS] 3. rare-cell coverage: min India_0 per outer val fold = 4 (at r0f0), min Italy_1 per outer val fold = 4 (at r0f0)
  [PASS] 4. pair counts (palpebral): pool n=184 cells={'India_0': 23, 'India_1': 57, 'Italy_0': 84, 'Italy_1': 20} | India 57x23=1311 pairs (32.8x the single split's 40) | Italy 20x84=1680 pairs (28.0x the single split's 60)

  Training 25 fits (5 repeats x 5 folds):
Downloading: "https://download.pytorch.org/models/densenet121-a639ec97.pth" to /root/.cache/torch/hub/che

In [18]:
# Step 1 -- 10/12: densenet121_forniceal_palpebral
!python classification/step1_cv_harness/run_cv_harness.py --combo densenet121_forniceal_palpebral_v2_clean
sync_outputs()


Step 1 harness: densenet121_forniceal_palpebral_v2_clean
  device=cuda  arch=densenet121  tissue=forniceal_palpebral
  locked hyperparameters (from v2_clean_scripts/outputs/densenet121_forniceal_palpebral_v2_clean/densenet121_forniceal_palpebral_v2_clean_study_summary.json): lr=0.0183628 wd=0.000153747 dropout=0.5

  Structural verification (checkpoints 1-4):
  [PASS] 1. test-set isolation: 0 of 33 held-out test patients appear in any fold
  [PASS] 2. partition integrity: each of 5 repeats partitions all 178 pool patients exactly once
  [PASS] 3. rare-cell coverage: min India_0 per outer val fold = 4 (at r0f0), min Italy_1 per outer val fold = 3 (at r0f4)
  [PASS] 4. pair counts (forniceal_palpebral): pool n=178 cells={'India_0': 23, 'India_1': 57, 'Italy_0': 79, 'Italy_1': 19} | India 57x23=1311 pairs (32.8x the single split's 40) | Italy 19x79=1501 pairs (25.0x the single split's 60)

  Training 25 fits (5 repeats x 5 folds):
    r0f0: stopped epoch 16 (best inner epoch 9, inner_los

In [19]:
# Step 1 -- 11/12: convnext_tiny_palpebral
!python classification/step1_cv_harness/run_cv_harness.py --combo convnext_tiny_palpebral_v2_clean
sync_outputs()


Step 1 harness: convnext_tiny_palpebral_v2_clean
  device=cuda  arch=convnext_tiny  tissue=palpebral
  locked hyperparameters (from v2_clean_scripts/outputs/convnext_tiny_palpebral_v2_clean/convnext_tiny_palpebral_v2_clean_study_summary.json): lr=0.000585494 wd=2.00941e-05 dropout=0.2

  Structural verification (checkpoints 1-4):
  [PASS] 1. test-set isolation: 0 of 33 held-out test patients appear in any fold
  [PASS] 2. partition integrity: each of 5 repeats partitions all 184 pool patients exactly once
  [PASS] 3. rare-cell coverage: min India_0 per outer val fold = 4 (at r0f0), min Italy_1 per outer val fold = 4 (at r0f0)
  [PASS] 4. pair counts (palpebral): pool n=184 cells={'India_0': 23, 'India_1': 57, 'Italy_0': 84, 'Italy_1': 20} | India 57x23=1311 pairs (32.8x the single split's 40) | Italy 20x84=1680 pairs (28.0x the single split's 60)

  Training 25 fits (5 repeats x 5 folds):
Downloading: "https://download.pytorch.org/models/convnext_tiny-983f1562.pth" to /root/.cache/tor

In [20]:
# Step 1 -- 12/12: convnext_tiny_forniceal_palpebral
!python classification/step1_cv_harness/run_cv_harness.py --combo convnext_tiny_forniceal_palpebral_v2_clean
sync_outputs()


Step 1 harness: convnext_tiny_forniceal_palpebral_v2_clean
  device=cuda  arch=convnext_tiny  tissue=forniceal_palpebral
  locked hyperparameters (from v2_clean_scripts/outputs/convnext_tiny_forniceal_palpebral_v2_clean/convnext_tiny_forniceal_palpebral_v2_clean_study_summary.json): lr=0.0773445 wd=0.000479821 dropout=0.5

  Structural verification (checkpoints 1-4):
  [PASS] 1. test-set isolation: 0 of 33 held-out test patients appear in any fold
  [PASS] 2. partition integrity: each of 5 repeats partitions all 178 pool patients exactly once
  [PASS] 3. rare-cell coverage: min India_0 per outer val fold = 4 (at r0f0), min Italy_1 per outer val fold = 3 (at r0f4)
  [PASS] 4. pair counts (forniceal_palpebral): pool n=178 cells={'India_0': 23, 'India_1': 57, 'Italy_0': 79, 'Italy_1': 19} | India 57x23=1311 pairs (32.8x the single split's 40) | Italy 19x79=1501 pairs (25.0x the single split's 60)

  Training 25 fits (5 repeats x 5 folds):
    r0f0: stopped epoch 39 (best inner epoch 32, 

## Checkpoint 5 -- label-shuffle negative control

The cheapest available proof that the harness does not leak. Given this project's history with
silent data bugs (the white-background convention, v1 template matching), this is the check
least worth skipping.

- **`within_country`** permutes labels inside each country, preserving each country's label rate
  (India ~71.6% anemic, Italy ~18.9%) while destroying every per-patient association.
  Per-country AUC **must** collapse to 0.50. Overall AUC may legitimately stay above 0.50 --
  which is a direct empirical demonstration of the country-shortcut mechanism, since 62% of the
  overall AUC's pairs are cross-country.
- **`global`** destroys the country-label association too, so *every* AUC must collapse to 0.50.
  A stricter pure-leakage test that cannot demonstrate the mechanism.

In [21]:
# Control A -- within-country permutation (leakage test + shortcut demonstration)
!python classification/step1_cv_harness/run_cv_harness.py \
    --combo mobilenet_v3_small_palpebral_v2_clean --shuffle-control within_country
sync_outputs()


Step 1 harness: mobilenet_v3_small_palpebral_v2_clean__shuffle_within_country
  device=cuda  arch=mobilenet_v3_small  tissue=palpebral
  locked hyperparameters (from v2_clean_scripts/outputs/mobilenet_v3_small_palpebral_v2_clean/mobilenet_v3_small_palpebral_v2_clean_study_summary.json): lr=0.0808856 wd=1.0423e-06 dropout=0.5
  NEGATIVE CONTROL: labels permuted (within_country) -- per-country AUC must collapse to 0.50

  Structural verification (checkpoints 1-4):
  [PASS] 1. test-set isolation: 0 of 33 held-out test patients appear in any fold
  [PASS] 2. partition integrity: each of 5 repeats partitions all 184 pool patients exactly once
  [PASS] 3. rare-cell coverage: min India_0 per outer val fold = 4 (at r0f0), min Italy_1 per outer val fold = 4 (at r0f0)
  [PASS] 4. pair counts (palpebral): pool n=184 cells={'India_0': 23, 'India_1': 57, 'Italy_0': 84, 'Italy_1': 20} | India 57x23=1311 pairs (32.8x the single split's 40) | Italy 20x84=1680 pairs (28.0x the single split's 60)

  Tr

In [22]:
# Control B -- global permutation (strict leakage test: every AUC must reach chance)
!python classification/step1_cv_harness/run_cv_harness.py \
    --combo mobilenet_v3_small_palpebral_v2_clean --shuffle-control global
sync_outputs()


Step 1 harness: mobilenet_v3_small_palpebral_v2_clean__shuffle_global
  device=cuda  arch=mobilenet_v3_small  tissue=palpebral
  locked hyperparameters (from v2_clean_scripts/outputs/mobilenet_v3_small_palpebral_v2_clean/mobilenet_v3_small_palpebral_v2_clean_study_summary.json): lr=0.0808856 wd=1.0423e-06 dropout=0.5
  NEGATIVE CONTROL: labels permuted (global) -- per-country AUC must collapse to 0.50

  Structural verification (checkpoints 1-4):
  [PASS] 1. test-set isolation: 0 of 33 held-out test patients appear in any fold
  [PASS] 2. partition integrity: each of 5 repeats partitions all 184 pool patients exactly once
  [PASS] 3. rare-cell coverage: min India_0 per outer val fold = 8 (at r0f2), min Italy_1 per outer val fold = 7 (at r0f4)
  [PASS] 4. pair counts (palpebral): pool n=184 cells={'India_0': 42, 'India_1': 38, 'Italy_0': 65, 'Italy_1': 39} | India 38x42=1596 pairs (39.9x the single split's 40) | Italy 39x65=2535 pairs (42.2x the single split's 60)

  Training 25 fits (

## Aggregate -- checkpoints 7, 8, 9

Writes the frozen Step 1 baseline (`outputs/baseline/step1_baseline.{csv,json,md}`) and evaluates
the remaining gates. Exits non-zero and refuses to declare Step 1 clear if any gate fails.

- **7 plausibility** -- pooled India AUC far outside what the single-split CIs already permitted
  means suspect the harness, not celebrate a discovery.
- **8 precision** -- India AUC 95% CI half-width <= 0.12. This is the actual pass/fail criterion:
  if it fails, Steps 3-5 cannot demonstrate anything and their success criteria must be
  renegotiated rather than quietly ignored.
- **9 artifact** -- the baseline file itself, recording seed, fold config and locked
  hyperparameters so later steps can run a valid *paired* comparison against it.

**Note: run standalone, this aggregates only the 12 CNN combos present in `outputs/` at this
point** -- it is a valid partial check (all 4 gates still apply and mean something), but it is
not the final Step 1 baseline. Get the full 18-combo baseline by downloading both this notebook's
and `step1-cv-harness-vit.ipynb`'s output zips, merging their `outputs/` folders locally into one
`classification/step1_cv_harness/outputs/`, and re-running `aggregate_baseline.py` there --
combo discovery is dynamic (globs `outputs/*/cv_metrics.json`), so no code change is needed.

In [23]:
!python classification/step1_cv_harness/aggregate_baseline.py
sync_outputs()


Step 1 baseline -- 12 combos
                                          combo  india_auc  india_ci_half_width  italy_auc       gap
     convnext_tiny_forniceal_palpebral_v2_clean   0.697483             0.134640   0.871286 -0.173803
               convnext_tiny_palpebral_v2_clean   0.718993             0.128146   0.842619 -0.123626
       densenet121_forniceal_palpebral_v2_clean   0.581083             0.143440   0.853564 -0.272481
                 densenet121_palpebral_v2_clean   0.586880             0.157141   0.819405 -0.232525
   efficientnet_b0_forniceal_palpebral_v2_clean   0.632342             0.152946   0.861959 -0.229617
             efficientnet_b0_palpebral_v2_clean   0.495957             0.180778   0.660000 -0.164043
mobilenet_v3_small_forniceal_palpebral_v2_clean   0.587643             0.170481   0.753498 -0.165855
          mobilenet_v3_small_palpebral_v2_clean   0.522197             0.148007   0.604286 -0.082089
    regnet_y_400mf_forniceal_palpebral_v2_clean   0.490008   

In [24]:
from pathlib import Path

report = Path('classification/step1_cv_harness/outputs/baseline/step1_baseline.md')
print(report.read_text(encoding='utf-8') if report.exists() else 'Baseline not produced -- check the aggregate cell above.')

# Step 1 Baseline -- Pooled Out-of-Fold Repeated Cross-Validation

Generated: 2026-08-08T18:27:53.868048+00:00

Frozen reference for Steps 2-5. Every later intervention must be compared against this
using a **paired** bootstrap on the difference (`cv_stats.paired_delta_auc`) with the
identical fold assignments recorded in each combo's `fold_manifest.json` -- not by
checking whether two independent confidence intervals overlap.

## Configuration

- Design: 5-fold x 5 repeats, stratified on country x label
- Seed: 42
- Bootstrap replicates: 2000
- Pool: train + val only; the 33-patient test split is sealed for Step 6
- Hyperparameters: each combo's own winning Optuna trial, reused verbatim (no re-tuning)

## Precision achieved

| | India pairs | Italy pairs |
|---|---|---|
| Single 70/15/15 split | 40 | 60 |
| Pooled out-of-fold | 1311 | 1680 |

## Results (sorted by India AUC)

| Combo | India AUC [95% CI] | Italy AUC [95% CI] | Overall AUC | Gap [95% CI] | Gap excl. 0 |
|---|---|---|--

## Done -- what to download

`/kaggle/working/step1_cv_results_cnn.zip` contains, per combo:
- `oof_predictions.csv` -- one held-out probability per patient per repeat (the pooled statistic's raw input)
- `fold_manifest.json` -- the exact fold assignments, which Steps 3-5 **must** reuse so their
  comparison against this baseline can be paired
- `cv_metrics.json` -- pooled AUC + bootstrap CIs, gate results
- `fold_metrics.csv` -- per-fold (own held-out set) accuracy/precision/recall/specificity/balanced
  accuracy/F1/AUC, one row per (repeat, fold)
- `fold_diagnostics.json` -- the raw per-fold loss histories + confusion matrices + ROC curves behind the plots below
- `plots/{combo}_loss_curves_grid.png` -- train vs. inner-val loss, one subplot per fold, single image
- `plots/{combo}_confusion_matrices_grid.png` -- confusion matrix per fold, single image
- `plots/{combo}_roc_curves_grid.png` -- ROC curve per fold, single image
- `plots/{combo}_fold_metrics_summary.png` -- AUC/F1/Balanced Accuracy across all folds, one chart

Plus `baseline/`, `comparison/` (cross-combo AUC and gap charts with bootstrap CI whiskers -- partial
until merged with the ViT notebook's results), and `structural_verification.json`.

Still no model checkpoints, by design -- Step 1 needs predictions and diagnostics, not weights.

**Next step after both notebooks finish:** download this zip and `step1-cv-harness-vit.ipynb`'s
zip, extract both `outputs/` into the same local `classification/step1_cv_harness/outputs/`, then
run `aggregate_baseline.py` once locally for the combined 18-combo baseline (and its own
combined `comparison/` charts).

In [25]:
from pathlib import Path

print('Final contents of /kaggle/working/outputs:')
for f in sorted(Path('/kaggle/working/outputs').rglob('*')):
    if f.is_file():
        print(f'  {f.relative_to("/kaggle/working/outputs")}  ({f.stat().st_size / 1e6:.3f} MB)')

zip_path = Path('/kaggle/working/step1_cv_results_cnn.zip')
print(f'\nZip archive: {zip_path}  ({zip_path.stat().st_size / 1e6:.2f} MB)')

Final contents of /kaggle/working/outputs:
  baseline/step1_baseline.csv  (0.005 MB)
  baseline/step1_baseline.json  (0.015 MB)
  baseline/step1_baseline.md  (0.005 MB)
  comparison/country_auc_comparison.png  (0.099 MB)
  comparison/india_italy_gap_comparison.png  (0.085 MB)
  convnext_tiny_forniceal_palpebral_v2_clean/cv_metrics.json  (0.008 MB)
  convnext_tiny_forniceal_palpebral_v2_clean/fold_diagnostics.json  (0.049 MB)
  convnext_tiny_forniceal_palpebral_v2_clean/fold_manifest.json  (0.175 MB)
  convnext_tiny_forniceal_palpebral_v2_clean/fold_metrics.csv  (0.003 MB)
  convnext_tiny_forniceal_palpebral_v2_clean/oof_predictions.csv  (0.039 MB)
  convnext_tiny_forniceal_palpebral_v2_clean/plots/convnext_tiny_forniceal_palpebral_v2_clean_confusion_matrices_grid.png  (0.092 MB)
  convnext_tiny_forniceal_palpebral_v2_clean/plots/convnext_tiny_forniceal_palpebral_v2_clean_fold_metrics_summary.png  (0.109 MB)
  convnext_tiny_forniceal_palpebral_v2_clean/plots/convnext_tiny_forniceal_palp